
# Databricks GenAI Production Lifecycle — Webinar Notebook

**Build → Evaluate → Deploy → Observe → Improve**  
**Govern & secure across the full lifecycle**

This notebook is designed as a compact webinar demo using the built-in **`samples.wanderbricks`** dataset. It creates a small RAG application over Wanderbricks property records and then walks the same application through the production lifecycle.


### Highlights

| Phase | What this notebook demonstrates |
|---|---|
| **Build** | Delta source table → AI Search index → retrieve → grounded LLM answer |
| **Evaluate** | MLflow evaluation dataset + RAG quality scorers |
| **Deploy** | MLflow model registration + optional Model Serving endpoint |
| **Observe** | MLflow traces and retriever / LLM spans |
| **Improve** | Change retrieval + prompt, rerun the same eval set, compare |
| **Govern** | Unity Catalog ownership, grants, lineage-friendly tables and versioned model assets |

**Current Databricks note (2026):** Databricks Apps is the recommended deployment path for new custom agents. This notebook uses Model Serving in the deployment section because it keeps the demo self-contained in one notebook and makes the deployment mechanics visible.



## 0. Prerequisites

You need a Unity Catalog-enabled Databricks workspace with:

- Access to `samples.wanderbricks`
- A writable Unity Catalog catalog/schema
- Databricks AI Search enabled (formerly Vector Search)
- Access to a Databricks-hosted chat model and embedding model
- Permission to create AI Search resources and Model Serving endpoints if you choose to provision them

The defaults below are examples. If your workspace exposes different Foundation Model endpoint names, change the notebook widgets before running the lifecycle.


In [0]:

%pip install -q --upgrade "mlflow[databricks]>=3.14.0" databricks-openai databricks-ai-search pandas
dbutils.library.restartPython()


In [0]:

# Configuration — change these values from the widget panel if needed.
dbutils.widgets.text("catalog_name", "main", "01 - Writable UC catalog")
dbutils.widgets.text("schema_name", "webinar_ai_lifecycle", "02 - Demo schema")
dbutils.widgets.text("ai_search_endpoint", "webinar-rag-search", "03 - AI Search endpoint")
dbutils.widgets.text("embedding_model", "databricks-qwen3-embedding-0-6b", "04 - Embedding endpoint/model")
dbutils.widgets.text("llm_model", "databricks-llama-4-maverick", "05 - Chat model endpoint/service")
dbutils.widgets.dropdown("create_ai_search_resources", "true", ["true", "false"], "06 - Create AI Search resources")
dbutils.widgets.dropdown("deploy_model_serving", "false", ["false", "true"], "07 - Actually deploy Model Serving")
dbutils.widgets.dropdown("use_llm_judges", "true", ["true", "false"], "08 - Use MLflow LLM judges")

CATALOG = dbutils.widgets.get("catalog_name")
SCHEMA = dbutils.widgets.get("schema_name")
AI_SEARCH_ENDPOINT = dbutils.widgets.get("ai_search_endpoint")
EMBEDDING_MODEL = dbutils.widgets.get("embedding_model")
LLM_MODEL = dbutils.widgets.get("llm_model")
CREATE_AI_SEARCH_RESOURCES = dbutils.widgets.get("create_ai_search_resources").lower() == "true"
DEPLOY_MODEL_SERVING = dbutils.widgets.get("deploy_model_serving").lower() == "true"
USE_LLM_JUDGES = dbutils.widgets.get("use_llm_judges").lower() == "true"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_documents"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_index"
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_model"
SERVING_ENDPOINT_NAME = "wanderbricks-rag-webinar"

print("SOURCE_TABLE          =", SOURCE_TABLE)
print("INDEX_NAME            =", INDEX_NAME)
print("LLM_MODEL             =", LLM_MODEL)
print("CREATE_SEARCH         =", CREATE_AI_SEARCH_RESOURCES)
print("DEPLOY_MODEL_SERVING  =", DEPLOY_MODEL_SERVING)


In [0]:
# Check the Databricks environment first
print("Available catalogs:")
display(spark.sql("SHOW CATALOGS"))

print("Current catalog/schema:")
display(
    spark.sql("""
        SELECT
            current_catalog() AS catalog,
            current_schema() AS schema
    """)
)

# Check whether Unity Catalog is enabled
try:
    display(spark.sql("SELECT current_metastore()"))
except Exception as e:
    print("Unity Catalog check failed:", e)


# 1. BUILD — Production RAG Reference Architecture

**Goal:** turn governed enterprise data into a small production-style RAG application.

Logical path:

**Wanderbricks source → governed Delta table → AI Search embeddings/index → retrieve → prompt → LLM → grounded answer**


In [0]:
# ============================================================
# BUILD — Cell 1
# Set up the Unity Catalog location and inspect Wanderbricks
# ============================================================

# Use the writable Unity Catalog available in this workspace
CATALOG = "workspace"
SCHEMA = "webinar_ai_lifecycle"

# Objects that later notebook cells will create/use
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_documents"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_index"
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.wanderbricks_rag_model"

print("Using catalog :", CATALOG)
print("Using schema  :", SCHEMA)
print("Source table  :", SOURCE_TABLE)
print("Search index  :", INDEX_NAME)

# ------------------------------------------------------------
# 1. Confirm the Wanderbricks sample dataset is available
# ------------------------------------------------------------

print("\nTables available in samples.wanderbricks:")
display(
    spark.sql("""
        SHOW TABLES IN samples.wanderbricks
    """)
)

# ------------------------------------------------------------
# 2. Create our own writable schema
# ------------------------------------------------------------

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
    """
)

print(f"\nSchema ready: {CATALOG}.{SCHEMA}")

# ------------------------------------------------------------
# 3. Switch notebook context to our working schema
# ------------------------------------------------------------

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("\nCurrent notebook context:")
display(
    spark.sql("""
        SELECT
            current_catalog() AS catalog,
            current_schema() AS schema
    """)
)

# ------------------------------------------------------------
# 4. Load the built-in Wanderbricks property dataset
# ------------------------------------------------------------

properties = spark.read.table(
    "samples.wanderbricks.properties"
)

print("\nWanderbricks property columns:")
print(properties.columns)

print("\nSample property records:")
display(properties.limit(5))

print("Number of available property records:", properties.count())

In [0]:

# Build a compact text corpus from Wanderbricks property rows.
# Using all columns makes the demo resilient to small schema changes in the sample table.

from pyspark.sql import functions as F

properties = spark.read.table("samples.wanderbricks.properties").limit(200)

text_parts = [
    F.concat(F.lit(f"{c}: "), F.coalesce(F.col(c).cast("string"), F.lit("null")))
    for c in properties.columns
]

docs = (
    properties
    .select(
        F.col("property_id").cast("string").alias("doc_id"),
        F.col("title").cast("string").alias("title"),
        F.concat_ws("\n", *text_parts).alias("text")
    )
    .dropna(subset=["doc_id", "text"])
)

(
    docs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SOURCE_TABLE)
)

# Standard AI Search endpoints require Change Data Feed on the source Delta table.
spark.sql(
    f"ALTER TABLE {SOURCE_TABLE} "
    "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)

display(spark.table(SOURCE_TABLE).limit(5))
print("RAG corpus rows:", spark.table(SOURCE_TABLE).count())


In [0]:
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient()

endpoints = search_client.list_endpoints()

for ep in endpoints.get("endpoints", []):
    print("Name:", ep.get("name"))
    print("Type:", ep.get("endpoint_type"))
    print("Status:", ep.get("endpoint_status"))
    print("-" * 50)

In [0]:
# ============================================================
# BUILD — Reuse existing AI Search endpoint
# ============================================================

from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(
    disable_notice=True
)

AI_SEARCH_ENDPOINT = "webinar-rag-search"

endpoint = search_client.get_endpoint(
    name=AI_SEARCH_ENDPOINT
)

print("Using AI Search endpoint:")
print("Name  :", AI_SEARCH_ENDPOINT)
print("Status:", endpoint.get("endpoint_status"))

In [0]:

# Build the RAG application with MLflow tracing.
# The retriever emits MLflow Document objects so RAG judges can inspect retrieved evidence.

import mlflow
from mlflow.entities import Document, SpanType
from databricks_openai import DatabricksOpenAI

mlflow.set_tracking_uri("databricks")
mlflow.openai.autolog()

llm_client = DatabricksOpenAI()

def _search_rows(question: str, k: int, query_type: str):
    idx = search_client.get_index(index_name=INDEX_NAME)
    return idx.similarity_search(
        query_text=question,
        columns=["doc_id", "title", "text"],
        num_results=k,
        query_type=query_type,
    )

@mlflow.trace(name="retrieve_wanderbricks", span_type=SpanType.RETRIEVER)
def retrieve_documents(question: str, k: int = 3, query_type: str = "ANN"):
    result = _search_rows(question, k=k, query_type=query_type)
    rows = result["result"]["data_array"]

    docs = []
    for row in rows:
        # AI Search appends the relevance score as the last value.
        doc_id, title, text, score = row[0], row[1], row[2], row[-1]
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "doc_uri": f"wanderbricks://property/{doc_id}",
                    "doc_id": str(doc_id),
                    "title": str(title),
                    "relevance_score": float(score),
                },
            )
        )

    # Explicitly set the retriever output in the MLflow retriever schema.
    active_span = mlflow.get_current_active_span()
    if active_span:
        active_span.set_outputs(docs)
    return docs

PROMPT_V1 = """You are a travel-property assistant.
Answer the user's question using only the retrieved property records.
If the evidence is insufficient, say that the available records do not contain enough information.
Cite supporting property records using [doc_id].

Question:
{question}

Retrieved evidence:
{context}
"""

PROMPT_V2 = """You are a precise enterprise RAG assistant.
Use ONLY facts explicitly present in the retrieved evidence.
Prefer exact field values over inference.
If evidence is missing or conflicting, explicitly say 'Insufficient evidence'.
Every factual statement must be supported by at least one citation in the form [doc_id].
Keep the answer concise.

Question:
{question}

Retrieved evidence:
{context}
"""

@mlflow.trace(name="wanderbricks_rag", span_type=SpanType.CHAIN)
def run_rag(question: str, k: int = 3, query_type: str = "ANN", prompt_version: str = "v1") -> str:
    retrieved = retrieve_documents(question, k=k, query_type=query_type)
    context = "\n\n".join(
        f"[{d.metadata['doc_id']}] {d.page_content}" for d in retrieved
    )

    template = PROMPT_V1 if prompt_version == "v1" else PROMPT_V2
    prompt = template.format(question=question, context=context)

    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Return grounded answers with source citations."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        max_tokens=300,
    )
    return response.choices[0].message.content

# Stable wrappers used in the evaluation phases.
def rag_v1(question: str) -> str:
    return run_rag(question, k=1, query_type="ANN", prompt_version="v1")

def rag_v2(question: str) -> str:
    return run_rag(question, k=4, query_type="HYBRID", prompt_version="v2")


In [0]:

# BUILD demo: one user question through retrieval + generation.
demo_question = "What type of property is available in the Wanderbricks listings? Give one example."
answer = rag_v1(demo_question)
print(answer)

print("\nOpen the MLflow trace from this notebook run to show:")
print("User question → retriever span → model call → grounded answer")



# 2. EVALUATE — GenAI Quality & Release Gate

**Goal:** prove that a candidate prompt/retriever/model is better, not merely newer.

We generate a tiny evaluation set dynamically from the same governed Wanderbricks data so the questions always match the sample data actually present in the workspace.


In [0]:

# Create 4 deterministic evaluation questions from real property rows.
eval_source = (
    spark.read.table("samples.wanderbricks.properties")
    .select("property_id", "title", "property_type")
    .dropna()
    .limit(4)
    .collect()
)

eval_data = []
for r in eval_source:
    eval_data.append({
        "inputs": {
            "question": f'What is the property type of "{r["title"]}"?'
        },
        "expectations": {
            "expected_property_type": str(r["property_type"]),
            "expected_facts": [
                f'{r["title"]} has property type {r["property_type"]}.'
            ],
        },
    })

eval_data


In [0]:
# ============================================================
# EVALUATE — Define scorers and run V1 evaluation
# ============================================================

import re
import mlflow
from mlflow.genai.scorers import scorer

# ------------------------------------------------------------
# 1. Put evaluation runs into one MLflow experiment
# ------------------------------------------------------------

mlflow.set_experiment(
    "/Shared/wanderbricks-rag-webinar-evaluation"
)

# ------------------------------------------------------------
# 2. Deterministic scorers
# ------------------------------------------------------------

@scorer
def citation_present(*, inputs=None, outputs=None,
                     expectations=None, trace=None):

    text = str(outputs or "")

    return bool(
        re.search(r"\[[^\]]+\]", text)
    )


@scorer
def expected_property_type_mentioned(
    *, inputs=None, outputs=None,
    expectations=None, trace=None
):

    expected = (
        expectations or {}
    ).get("expected_property_type")

    if expected is None:
        return False

    return (
        str(expected).lower()
        in str(outputs or "").lower()
    )


scorers_v1 = [
    citation_present,
    expected_property_type_mentioned
]

# ------------------------------------------------------------
# 3. Optional MLflow LLM-as-a-Judge scorers
# ------------------------------------------------------------

if USE_LLM_JUDGES:

    from mlflow.genai.scorers import (
        Correctness,
        RelevanceToQuery,
        RetrievalGroundedness,
        RetrievalRelevance,
    )

    scorers_v1 += [
        Correctness(),
        RelevanceToQuery(),
        RetrievalGroundedness(),
        RetrievalRelevance(),
    ]

# ------------------------------------------------------------
# 4. Run evaluation
# ------------------------------------------------------------

with mlflow.start_run(
    run_name="wanderbricks-rag-v1-evaluation"
):

    eval_v1 = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=rag_v1,
        scorers=scorers_v1,
    )

# ------------------------------------------------------------
# 5. Show results
# ------------------------------------------------------------

print("V1 evaluation run:", eval_v1.run_id)

print("\nV1 aggregate metrics:")

for metric_name, metric_value in eval_v1.metrics.items():
    print(
        f"{metric_name}: {metric_value}"
    )


# 3. DEPLOY — From Notebook PoC to a Governed Runtime

**Goal:** package the application, version it in Unity Catalog, and optionally expose it through a serving endpoint.

For a live webinar, you can leave `deploy_model_serving=false` and show the registration/versioning code without waiting for endpoint provisioning. Set it to `true` for a full deployment.

> For new custom agents, Databricks Apps is the recommended production deployment path. Model Serving remains useful for illustrating a notebook-to-endpoint workflow and for custom models.


In [0]:
# ============================================================
# 3. DEPLOY — Configuration
# ============================================================

import mlflow
import pandas as pd

REGISTERED_MODEL_NAME = (
    f"{CATALOG}.{SCHEMA}.wanderbricks_rag_model"
)

SERVING_ENDPOINT_NAME = "wanderbricks-rag-webinar"

# Keep False for the webinar preparation.
# Change to True only when you want to create a real serving endpoint.
DEPLOY_MODEL_SERVING = False

print("Registered model :", REGISTERED_MODEL_NAME)
print("Serving endpoint :", SERVING_ENDPOINT_NAME)
print("Deploy endpoint  :", DEPLOY_MODEL_SERVING)

In [0]:
# ============================================================
# 3. DEPLOY — Package the RAG application as MLflow PyFunc
# ============================================================

import mlflow.pyfunc


class WebinarRAGModel(mlflow.pyfunc.PythonModel):

    def __init__(
        self,
        index_name,
        ai_search_endpoint,
        llm_model
    ):
        self.index_name = index_name
        self.ai_search_endpoint = ai_search_endpoint
        self.llm_model = llm_model

    def predict(
        self,
        context,
        model_input,
        params=None
    ):

        from databricks.ai_search.client import AISearchClient
        from databricks_openai import DatabricksOpenAI

        search_client = AISearchClient(
            disable_notice=True
        )

        llm_client = DatabricksOpenAI()

        index = search_client.get_index(
            endpoint_name=self.ai_search_endpoint,
            index_name=self.index_name
        )

        # Accept DataFrame input
        if isinstance(model_input, pd.DataFrame):

            questions = (
                model_input["question"]
                .astype(str)
                .tolist()
            )

        elif isinstance(model_input, list):

            questions = [
                str(q)
                for q in model_input
            ]

        else:

            questions = [
                str(model_input)
            ]

        answers = []

        for question in questions:

            result = index.similarity_search(
                query_text=question,
                columns=[
                    "doc_id",
                    "title",
                    "text"
                ],
                num_results=4,
                query_type="HYBRID"
            )

            rows = result["result"]["data_array"]

            evidence = "\n\n".join(
                [
                    f"[{row[0]}] {row[2]}"
                    for row in rows
                ]
            )

            prompt = f"""
You are a grounded enterprise RAG assistant.

Use ONLY the evidence below.

If the answer cannot be determined from the evidence,
say "Insufficient evidence".

Every factual statement should contain a citation
using [doc_id].

QUESTION:
{question}

EVIDENCE:
{evidence}
"""

            response = (
                llm_client.chat.completions.create(
                    model=self.llm_model,
                    messages=[
                        {
                            "role": "system",
                            "content":
                            "Answer only from retrieved evidence."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    temperature=0.1,
                    max_tokens=300
                )
            )

            answer = (
                response
                .choices[0]
                .message
                .content
            )

            answers.append(answer)

        return answers


print("WebinarRAGModel class created.")

In [0]:
# ============================================================
# 3. DEPLOY — Log + register model in Unity Catalog
# ============================================================

from mlflow.models import infer_signature

from mlflow.models.resources import (
    DatabricksVectorSearchIndex,
    DatabricksServingEndpoint
)


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")


input_example = pd.DataFrame(
    {
        "question": [
            demo_question
        ]
    }
)


output_example = [
    "Example grounded response [property-id]"
]


signature = infer_signature(
    input_example,
    output_example
)


resources = [

    DatabricksVectorSearchIndex(
        index_name=INDEX_NAME
    ),

    DatabricksServingEndpoint(
        endpoint_name=LLM_MODEL
    )
]


with mlflow.start_run(
    run_name="wanderbricks-rag-deploy"
):

    logged_model = mlflow.pyfunc.log_model(

        name="wanderbricks_rag",

        python_model=WebinarRAGModel(
            index_name=INDEX_NAME,
            ai_search_endpoint=AI_SEARCH_ENDPOINT,
            llm_model=LLM_MODEL
        ),

        signature=signature,

        input_example=input_example,

        resources=resources,

        pip_requirements=[
            "mlflow[databricks]>=3.14.0",
            "databricks-ai-search",
            "databricks-openai",
            "pandas"
        ],

        registered_model_name=REGISTERED_MODEL_NAME
    )


print("Model logged successfully")
print("Model URI :", logged_model.model_uri)
print("Model ID  :", logged_model.model_id)

In [0]:
# ============================================================
# 3. DEPLOY — Get latest registered model version
# ============================================================

from mlflow import MlflowClient


mlflow.set_registry_uri("databricks-uc")

uc_client = MlflowClient()


versions = list(
    uc_client.search_model_versions(
        filter_string=(
            f"name='{REGISTERED_MODEL_NAME}'"
        )
    )
)


if len(versions) == 0:

    raise RuntimeError(
        "No registered model version found."
    )


latest_version = max(
    versions,
    key=lambda v: int(v.version)
)


MODEL_VERSION = str(
    latest_version.version
)


print(
    "Registered model :",
    REGISTERED_MODEL_NAME
)

print(
    "Model version    :",
    MODEL_VERSION
)

In [0]:
# ============================================================
# 3. DEPLOY — OPTIONAL real Model Serving deployment
# ============================================================

if DEPLOY_MODEL_SERVING:

    from databricks.sdk import WorkspaceClient

    from databricks.sdk.service.serving import (
        EndpointCoreConfigInput,
        ServedEntityInput
    )


    w = WorkspaceClient()


    try:

        existing_endpoint = (
            w.serving_endpoints.get(
                name=SERVING_ENDPOINT_NAME
            )
        )

        print(
            "Serving endpoint already exists:",
            SERVING_ENDPOINT_NAME
        )

        print(
            "State:",
            existing_endpoint.state
        )


    except Exception:

        print(
            "Creating serving endpoint..."
        )

        endpoint = (
            w.serving_endpoints.create_and_wait(

                name=SERVING_ENDPOINT_NAME,

                config=EndpointCoreConfigInput(

                    served_entities=[

                        ServedEntityInput(

                            entity_name=
                            REGISTERED_MODEL_NAME,

                            entity_version=
                            MODEL_VERSION,

                            workload_size="Small",

                            scale_to_zero_enabled=True
                        )

                    ]
                )
            )
        )

        print(
            "Endpoint ready:",
            endpoint.name
        )

else:

    print(
        "Serving endpoint creation skipped."
    )

    print(
        "Model is registered and ready for deployment."
    )


# 4. OBSERVE — Trace the Black Box

**Goal:** turn one RAG answer into inspectable execution steps.

The same request should reveal:

**question → retrieval span → model span → final answer**

This is the operational bridge between “the answer looks wrong” and “we know exactly where it failed.”


In [0]:
# ============================================================
# 4. OBSERVE — Generate a production-style RAG trace
# ============================================================

import mlflow


mlflow.openai.autolog()


observe_question = (
    "What type of property is available "
    "in the Wanderbricks listings?"
)


observe_answer = rag_v1(
    observe_question
)


print("QUESTION")
print(observe_question)

print("\nANSWER")
print(observe_answer)


trace_id = (
    mlflow.get_last_active_trace_id()
)


print("\nTrace ID:")
print(trace_id)

In [0]:
# ============================================================
# 4. OBSERVE — Retrieve trace details
# ============================================================

if trace_id is None:

    raise RuntimeError(
        "No MLflow trace was generated."
    )


trace = mlflow.get_trace(
    trace_id,
    flush=True
)


print(
    "Trace retrieved:",
    trace.info.trace_id
)

In [0]:
# ============================================================
# 4. OBSERVE — Show span-level observability
# ============================================================

span_rows = []


for span in trace.data.spans:

    start_ns = getattr(
        span,
        "start_time_ns",
        None
    )

    end_ns = getattr(
        span,
        "end_time_ns",
        None
    )


    if (
        start_ns is not None
        and end_ns is not None
    ):

        duration_ms = round(
            (
                end_ns
                - start_ns
            )
            / 1_000_000,
            2
        )

    else:

        duration_ms = None


    span_rows.append(
        {
            "span_name":
                getattr(
                    span,
                    "name",
                    None
                ),

            "span_type":
                str(
                    getattr(
                        span,
                        "span_type",
                        ""
                    )
                ),

            "duration_ms":
                duration_ms,

            "status":
                str(
                    getattr(
                        span,
                        "status",
                        ""
                    )
                )
        }
    )


span_df = pd.DataFrame(
    span_rows
)


display(span_df)

In [0]:
# ============================================================
# 4. OBSERVE — Find retriever spans
# ============================================================

from mlflow.entities import SpanType


retriever_spans = (
    trace.search_spans(
        span_type=SpanType.RETRIEVER
    )
)


print(
    "Retriever spans found:",
    len(retriever_spans)
)


for span in retriever_spans:

    print(
        "\nRetriever:",
        span.name
    )

    print(
        "Inputs:",
        span.inputs
    )

    print(
        "Outputs:",
        span.outputs
    )

In [0]:
# ============================================================
# 5. IMPROVE — Compare V1 and V2
# ============================================================

test_question = (
    eval_data[0]["inputs"]["question"]
)


print("QUESTION")
print(test_question)


print(
    "\n======================"
)

print(
    "V1 — ANN / k=1 / Basic Prompt"
)

print(
    "======================"
)

v1_answer = rag_v1(
    test_question
)

print(v1_answer)


print(
    "\n======================"
)

print(
    "V2 — HYBRID / k=4 / Improved Prompt"
)

print(
    "======================"
)

v2_answer = rag_v2(
    test_question
)

print(v2_answer)


# 5. IMPROVE — Close the Production Feedback Loop

**Goal:** make one controlled change, rerun the same test evidence, and prove whether quality improved.

For the demo:

- **V1** = ANN retrieval, `k=1`, basic prompt
- **V2** = Hybrid retrieval, `k=4`, stricter evidence/citation prompt

The important point is not that V2 must always win. The point is that **improvement is measured against the same evaluation evidence before promotion**.


In [0]:
# ============================================================
# 5. IMPROVE — Define evaluation scorers for V2
# ============================================================

scorers_v2 = [

    citation_present,

    expected_property_type_mentioned

]


if USE_LLM_JUDGES:

    from mlflow.genai.scorers import (
        Correctness,
        RelevanceToQuery,
        RetrievalGroundedness,
        RetrievalRelevance
    )


    scorers_v2 += [

        Correctness(),

        RelevanceToQuery(),

        RetrievalGroundedness(),

        RetrievalRelevance()

    ]


print(
    "Number of V2 scorers:",
    len(scorers_v2)
)

In [0]:
# ============================================================
# 5. IMPROVE — Evaluate improved RAG
# ============================================================

mlflow.set_experiment(
    "/Shared/wanderbricks-rag-webinar-evaluation"
)


with mlflow.start_run(
    run_name=
    "wanderbricks-rag-v2-evaluation"
):

    eval_v2 = (
        mlflow.genai.evaluate(

            data=eval_data,

            predict_fn=rag_v2,

            scorers=scorers_v2

        )
    )


print(
    "V2 evaluation run:",
    eval_v2.run_id
)


print(
    "\nV2 aggregate metrics:"
)


for (
    metric_name,
    metric_value
) in eval_v2.metrics.items():

    print(
        f"{metric_name}: "
        f"{metric_value}"
    )

In [0]:
# ============================================================
# 5. IMPROVE — Side-by-side evaluation comparison
# ============================================================

all_metric_names = sorted(

    set(
        eval_v1.metrics.keys()
    )

    |

    set(
        eval_v2.metrics.keys()
    )

)


comparison_rows = []


for metric in all_metric_names:

    v1_value = (
        eval_v1.metrics.get(
            metric
        )
    )

    v2_value = (
        eval_v2.metrics.get(
            metric
        )
    )


    comparison_rows.append(
        {
            "Metric":
                metric,

            "V1":
                v1_value,

            "V2":
                v2_value
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


display(
    comparison_df
)

In [0]:
# ============================================================
# 5. IMPROVE — Simple release decision
# ============================================================

def numeric_metric_average(
    metrics
):

    values = []

    for value in metrics.values():

        if isinstance(
            value,
            (int, float)
        ):

            values.append(
                float(value)
            )

    if len(values) == 0:

        return None

    return sum(values) / len(values)


v1_average = (
    numeric_metric_average(
        eval_v1.metrics
    )
)

v2_average = (
    numeric_metric_average(
        eval_v2.metrics
    )
)


print(
    "V1 metric average:",
    v1_average
)

print(
    "V2 metric average:",
    v2_average
)


if (
    v1_average is not None
    and
    v2_average is not None
):

    if v2_average >= v1_average:

        print(
            "\nRELEASE DECISION:"
        )

        print(
            "V2 passes the simple "
            "quality comparison."
        )

    else:

        print(
            "\nRELEASE DECISION:"
        )

        print(
            "Keep V1 and investigate "
            "V2 regression."
        )


# 6. GOVERN — Cross-Cutting Control Layer

Governance is **not a final step after observability**. It spans the complete lifecycle.

In this demo:

- Source and derived RAG tables live in Unity Catalog
- The AI Search index is tied to a governed Delta source
- Model/app versions are registered in Unity Catalog
- Permissions can be inspected explicitly
- Table history provides change evidence
- MLflow provides evaluation and tracing evidence around the application lifecycle

Use this section to make the point that **Build, Evaluate, Deploy, Observe and Improve are governed continuously**.


In [0]:
# ============================================================
# 6. GOVERN — Unity Catalog table governance
# ============================================================

print(
    "SOURCE TABLE:"
)

print(
    SOURCE_TABLE
)


print(
    "\nTABLE GRANTS"
)


display(

    spark.sql(
        f"""
        SHOW GRANTS
        ON TABLE {SOURCE_TABLE}
        """
    )

)

In [0]:
# ============================================================
# 6. GOVERN — Data history / audit trail
# ============================================================

display(

    spark.sql(
        f"""
        DESCRIBE HISTORY
        {SOURCE_TABLE}
        """
    ).limit(20)

)

In [0]:
# ============================================================
# 6. GOVERN — Governed Delta table details
# ============================================================

display(

    spark.sql(
        f"""
        DESCRIBE DETAIL
        {SOURCE_TABLE}
        """
    )

)

In [0]:
# ============================================================
# 6. GOVERN — Registered model versions
# ============================================================

import mlflow

from mlflow import MlflowClient


mlflow.set_registry_uri(
    "databricks-uc"
)


uc_client = MlflowClient()


model_versions = list(

    uc_client.search_model_versions(

        filter_string=
        f"name='{REGISTERED_MODEL_NAME}'"

    )

)


model_governance_rows = []


for version in model_versions:

    model_governance_rows.append(
        {

            "model_name":
                version.name,

            "version":
                version.version,

            "status":
                str(
                    getattr(
                        version,
                        "status",
                        ""
                    )
                ),

            "created":
                getattr(
                    version,
                    "creation_timestamp",
                    None
                ),

            "source":
                getattr(
                    version,
                    "source",
                    None
                )
        }
    )


model_governance_df = (
    pd.DataFrame(
        model_governance_rows
    )
)


display(
    model_governance_df
)

In [0]:
# ============================================================
# ENTERPRISE GENAI PRODUCTION LIFECYCLE
# ============================================================

print(
    """
==========================================================

        ENTERPRISE GENAI PRODUCTION LIFECYCLE

 BUILD
   ↓
 EVALUATE
   ↓
 DEPLOY
   ↓
 OBSERVE
   ↓
 IMPROVE
   ↺

 GOVERN & SECURE ACROSS THE COMPLETE LIFECYCLE

==========================================================
"""
)